In [12]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier,
                               GradientBoostingClassifier, RandomForestClassifier)
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

sns.set_theme(style="whitegrid")

PROJECT_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR    = PROJECT_ROOT / "models"

# All feature importance plots go here
# All feature importance plots go here
FI_ROOT = PROJECT_ROOT / "outputs" / "feature_engineering_feature_selection"

# Force one subfolder per experiment inside the correct parent directory
FI_DIRS = {
    "with_FE_baseline"    : FI_ROOT / "with_FE_baseline",
    "with_FE_tuned"       : FI_ROOT / "with_FE_tuned",
    "without_FE_baseline" : FI_ROOT / "without_FE_baseline",
    "without_FE_tuned"    : FI_ROOT / "without_FE_tuned",
}

# Create the nested subfolders if they don't exist
for d in FI_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Feature importance output root successfully targeted at:", FI_ROOT)


Feature importance output root successfully targeted at: c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\feature_engineering_feature_selection


In [13]:
# with FE  (selected / engineered features)
X_train_fe = pd.read_csv(PROCESSED_DIR / "X_train_selected_imp.csv")
X_test_fe  = pd.read_csv(PROCESSED_DIR / "X_test_selected_imp.csv")

# without FE  (raw processed features)
X_train_no = pd.read_csv(PROCESSED_DIR / "X_train_processed.csv")
X_test_no  = pd.read_csv(PROCESSED_DIR / "X_test_processed.csv")

y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv")["PCOS"]
y_test  = pd.read_csv(PROCESSED_DIR / "y_test.csv")["PCOS"]

print("with FE    train:", X_train_fe.shape, " test:", X_test_fe.shape)
print("without FE train:", X_train_no.shape, " test:", X_test_no.shape)

with FE    train: (378, 11)  test: (163, 11)
without FE train: (378, 41)  test: (163, 41)


In [14]:
def plot_feature_importance(importances, feature_names, model_name, experiment_label, out_dir):
    """
    Bar chart of feature importances (or permutation importances).
    Sorted descending. Saved to out_dir/feature_importance_<model_name>.png
    """
    idx   = np.argsort(importances)[::-1]
    names = [feature_names[i] for i in idx]
    vals  = importances[idx]

    n = len(names)
    fig_h = max(4, n * 0.35)
    fig, ax = plt.subplots(figsize=(9, fig_h))

    colors = plt.cm.Blues(np.linspace(0.4, 0.9, n))[::-1]
    bars = ax.barh(range(n), vals[::-1], color=colors[::-1])
    ax.set_yticks(range(n))
    ax.set_yticklabels(names[::-1], fontsize=9)
    ax.set_xlabel("Importance Score", fontsize=10)
    ax.set_title(
        f"{experiment_label} — {model_name}\nFeature Importances",
        fontsize=11, fontweight="bold"
    )

    # Annotate values
    for bar, val in zip(bars, vals[::-1]):
        ax.text(bar.get_width() + max(vals) * 0.01, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=8, color="dimgray")

    plt.tight_layout()
    safe = (model_name.lower()
            .replace(" ", "_").replace("/", "_")
            .replace("(", "").replace(")", ""))
    path = out_dir / f"feature_importance_{safe}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")


def get_importances(model, X_train, y_train, X_test, feature_names):
    """
    Returns importance array.
    Priority: feature_importances_ → coef_ → permutation importance.
    """
    # Tree-based / boosting
    base = model
    # unwrap CalibratedClassifierCV
    if hasattr(model, "estimator"):
        base = model.estimator
    if hasattr(model, "calibrated_classifiers_"):
        try:
            base = model.calibrated_classifiers_[0].estimator
        except Exception:
            pass

    if hasattr(base, "feature_importances_"):
        return np.array(base.feature_importances_)

    if hasattr(base, "coef_"):
        coef = np.array(base.coef_)
        return np.abs(coef).mean(axis=0) if coef.ndim > 1 else np.abs(coef)

    # Fallback: permutation importance on test set
    result = permutation_importance(model, X_test, y_test,
                                    n_repeats=10, random_state=42, n_jobs=1)
    return result.importances_mean

print("Feature importance functions defined.")

Feature importance functions defined.


In [15]:
# Each entry: (experiment_key, X_train, X_test, model_prefix_in_filename)
# We load already-fitted models that were saved in notebooks 1-4.

experiments = [
    ("with_FE_baseline",    X_train_fe, X_test_fe,  "with_fe_baseline"),
    ("with_FE_tuned",       X_train_fe, X_test_fe,  "with_fe_tuned"),
    ("without_FE_baseline", X_train_no, X_test_no,  "without_fe_baseline"),
    ("without_FE_tuned",    X_train_no, X_test_no,  "without_fe_tuned"),
]

MODEL_NAMES = [
    "Logistic Regression", "Decision Tree", "Random Forest", "SVM",
    "Gaussian NB", "Bagging", "AdaBoost", "Gradient Boosting",
    "KNN", "LDA", "QDA", "Perceptron", "XGBoost", "Stacking ML",
]

def safe_name(name):
    return (name.lower()
            .replace(" ", "_").replace("/", "_")
            .replace("(", "").replace(")", "")
            .replace("+", "plus").replace(":", ""))

print("Configuration ready.")
print("Experiments:", [e[0] for e in experiments])

Configuration ready.
Experiments: ['with_FE_baseline', 'with_FE_tuned', 'without_FE_baseline', 'without_FE_tuned']


In [16]:
for exp_key, X_tr, X_te, prefix in experiments:
    # This grabs the corrected nested path from FI_DIRS
    out_dir = FI_DIRS[exp_key]
    feat_names = list(X_tr.columns)
    print(f"\n{'='*60}")
    print(f"Experiment : {exp_key}  ({len(feat_names)} features)")
    print(f"{'='*60}")

    for model_name in MODEL_NAMES:
        safe = safe_name(model_name)
        model_path = MODELS_DIR / f"{prefix}_{safe}.joblib"

        if not model_path.exists():
            # Try the tuned suffix variant
            model_path = MODELS_DIR / f"{prefix}_{safe}_tuned_model.joblib"

        if not model_path.exists():
            print(f"  ⚠ Model file not found: {model_path.name} — skipping")
            continue

        try:
            model = joblib.load(model_path)
            
            # FIXED: Pass y_train explicitly using the correct keyword argument 'y_train'
            imps  = get_importances(model, X_tr, y_train=y_train,
                                    X_test=X_te, feature_names=feat_names)

            # Safety check: importance length must match feature count
            if len(imps) != len(feat_names):
                print(f"  ⚠ Importance length mismatch for {model_name} "
                      f"({len(imps)} vs {len(feat_names)}) — using permutation")
                result = permutation_importance(model, X_te, y_test,
                                                n_repeats=10, random_state=42, n_jobs=1)
                imps = result.importances_mean

            # Save the figure into the explicit nested path
            plot_feature_importance(imps, feat_names, model_name, exp_key, out_dir)

        except Exception as e:
            print(f"  ✗ Error processing feature importances for {model_name}: {e}")

print("\n✓ All feature importance plots successfully generated.")
print("Verify your folder contents under:", FI_ROOT)



Experiment : with_FE_baseline  (11 features)
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\feature_engineering_feature_selection\with_FE_baseline\feature_importance_logistic_regression.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\feature_engineering_feature_selection\with_FE_baseline\feature_importance_decision_tree.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\feature_engineering_feature_selection\with_FE_baseline\feature_importance_random_forest.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\feature_engineering_feature_selection\with_FE_baseline\feature_importance_svm.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\feature_engineering_feature_selection\with_FE_baseline\feature_importance_gaussian_nb